# 11. Async Subagents — Run long tasks in the background

## Learning goals
- Understand why synchronous subagents block long-running work
- Define **non-blocking background work** with `AsyncSubAgent`
- Recognize the five task-management tools exposed to the supervisor
- Choose between ASGI co-deployment and remote HTTP deployment
- Expose background-task lifecycle events with Event Streaming v3
- Avoid common pitfalls such as immediate polling and truncated task IDs

> **Preview feature** — requires `deepagents>=0.5.0`. The API may change.

## Prerequisites

Async subagents require an **Agent Protocol server** such as a local `langgraph dev` runtime or a LangSmith Deployment. A plain Python session can define the graph, but it cannot actually run background tasks by itself.

- Local: `langgraph dev --n-jobs-per-worker 10`
- Remote: LangSmith Deployment or another Agent Protocol server

This notebook focuses on definitions and design. After scaffolding `langgraph.json` and graph files, run the app from a terminal with `langgraph dev` to exercise the conversation end to end.

In [ ]:
# Environment setup
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"),     "Set ANTHROPIC_API_KEY or OPENAI_API_KEY in your environment."
print("Environment ready")

---
## 1. Why async subagents?

A synchronous subagent blocks the parent until the delegated work returns.

| Scenario | Synchronous | Asynchronous |
|----------|-------------|--------------|
| Sub-10-second task | ✅ simple | unnecessary overhead |
| Multi-minute research or coding | ❌ frozen UX | ✅ background execution |
| Several research/coding lanes | ❌ sequential | ✅ concurrent |
| Mid-flight update or cancellation | ❌ unavailable | ✅ possible |
| Keep chatting while work runs | ❌ | ✅ |

Use async subagents when a task is long-running, parallel, or likely to need steering. For short tasks, regular subagents remain simpler.

---
## 2. Define an `AsyncSubAgent`

`AsyncSubAgent(...)` builds the dict spec Deep Agents consumes. For inspection and debugging, access fields with keys such as `subagent["graph_id"]`, not attributes.

| Field | Type | Required | Meaning |
|------|------|----------|---------|
| `name` | str | ✅ | Unique name used by the supervisor |
| `description` | str | ✅ | Delegation hint shown to the supervisor |
| `graph_id` | str | ✅ | Graph/assistant ID registered in the Agent Protocol server |
| `url` | str | ❌ | Remote server endpoint; omitted means ASGI co-deployment |
| `headers` | dict | ❌ | Authentication headers for a remote server |

In [ ]:
from deepagents import AsyncSubAgent, create_deep_agent

researcher = AsyncSubAgent(
    name="researcher",
    description="Performs long-running research and synthesis.",
    graph_id="researcher",
)

coder = AsyncSubAgent(
    name="coder",
    description="Handles longer coding and review tasks.",
    graph_id="coder",
    url="https://coder-deployment.langsmith.dev",
)

print(f"researcher -> graph_id={researcher['graph_id']}, url={researcher.get('url') or 'ASGI co-deployed'}")
print(f"coder      -> graph_id={coder['graph_id']}, url={coder.get('url')}")

---
## 3. Build the supervisor agent

The supervisor is still a normal `create_deep_agent()` graph. The important part is the system prompt: it must teach the model how to provide a good long-running-task UX.

In [ ]:
SUPERVISOR_PROMPT = 'You are a project coordinator.\nDelegate long-running work to async subagents and immediately return control to the user.\n\nRules:\n- After starting async work, do not immediately call `check_async_task`.\n- Only call `check_async_task` or `list_async_tasks` when the user asks for status.\n- Never shorten a `task_id`; always show the full ID.\n- Treat prior status in chat history as stale; re-check when fresh status matters.\n'

supervisor = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    system_prompt=SUPERVISOR_PROMPT,
    subagents=[researcher, coder],
)

print("Supervisor created (async subagents: researcher, coder)")

---
## 4. The five tools exposed to the supervisor

Registering `AsyncSubAgent` specs injects five task-management tools.

| Tool | Behavior | When to call |
|------|----------|--------------|
| `start_async_task` | Starts a subagent in the background and immediately returns `task_id` | A long-running request arrives |
| `check_async_task` | Reads task status; returns final output when complete | The user asks about a specific task |
| `update_async_task` | Sends a new instruction into the same running thread | Scope or direction changes mid-flight |
| `cancel_async_task` | Sends a cancel signal and marks the task cancelled | The task is no longer needed |
| `list_async_tasks` | Lists all tracked tasks and live/cached status | The user asks for the whole workboard |

Task metadata lives in a dedicated **`async_tasks` channel**, separate from message history, so it survives context compaction.

---
## 5. Scaffold `langgraph.json` for ASGI co-deployment

If `url` is omitted, the supervisor and subagents are deployed together in the same LangGraph server. Register every graph in one `langgraph.json`.

In [ ]:
# This cell writes an example file in the repository root.
# Adjust paths for a real project.
import json
from pathlib import Path

scaffold = {
    "dependencies": ["."],
    "graphs": {
        "supervisor": "./src/supervisor.py:graph",
        "researcher": "./src/researcher.py:graph",
        "coder": "./src/coder.py:graph",
    },
    "env": ".env",
}

scaffold_path = Path("langgraph.example.json")
scaffold_path.write_text(json.dumps(scaffold, indent=2))
print(f"Example file written: {scaffold_path.resolve()}")
print(json.dumps(scaffold, indent=2))

### Local run

```bash
uv add deepagents langchain-anthropic langgraph
langgraph dev --n-jobs-per-worker 10
```

A supervisor plus three concurrent subagent tasks needs at least four worker slots.

### Transport choice

| Transport | How | Pros | Cons |
|-----------|-----|------|------|
| ASGI co-deploy | omit `url` | in-process, low latency, single deployment | subagents cannot scale independently |
| HTTP remote | set `url` | independent scale, remote maintenance, org boundary | network latency, auth required |

Use `LANGSMITH_API_KEY` or `LANGGRAPH_API_KEY` for HTTP authentication.

---
## 6. Typical conversation flow

User: *“Research the latest AI agent framework trends.”*

1. The supervisor calls `start_async_task(name="researcher", instruction=...)`
2. The server immediately returns `task_id=abc123...`
3. The supervisor **does not check immediately**; it gives control back to the user
4. The user can continue chatting
5. Later: *“How is the researcher doing?”* → supervisor calls `check_async_task(task_id="abc123...")`
6. If still running, report status; if complete, summarize the output

### Mid-flight steering

- *“Ask the researcher to include Korean enterprise examples.”* → `update_async_task(...)`
- *“Stop that task.”* → `cancel_async_task(...)`

---
## 7. Observe background work with v2 unified streaming

The supervisor receives main-agent and subagent events through one stream call:

```python
agent.stream(..., stream_mode=..., subgraphs=True, version="v2")
```

| Signal | Detection | Meaning |
|--------|-----------|---------|
| Pending | main graph emits a task tool call | task is being requested |
| Running | first event arrives under a `tools:<id>` namespace | subagent is active |
| Complete | tool result returns to the main tools node | result is collected |

Changes in the `async_tasks` channel appear inside `stream_mode="updates"` data, including `check_async_task` and `list_async_tasks` results.

In [ ]:
# Async subagent v2 streaming pattern
streaming_example = """
for chunk in supervisor.stream(
    {"messages": [{"role": "user", "content": "Research the market deeply."}]},
    stream_mode=["updates", "messages", "custom"],
    subgraphs=True,
    version="v2",
):
    t = chunk["type"]
    is_subagent = any(seg.startswith("tools:") for seg in chunk["ns"])

    if t == "updates":
        for node_name, node_data in chunk["data"].items():
            print(f"[{'sub' if is_subagent else 'main'}] {node_name}")
    elif t == "messages":
        token, metadata = chunk["data"]
        if metadata.get("lc_source") == "summarization":
            continue
    elif t == "custom":
        print("custom:", chunk["data"])
"""
print(streaming_example)

---
## 8. Common pitfalls and fixes

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Immediate polling | `start_async_task` followed by `check_async_task` freezes UX | Put “return control immediately” in the system prompt |
| Truncated task ID | model stores `abc12...`; future lookup fails | Tell the model never to shorten `task_id` |
| Stale status | old status is reported as fresh | Re-check status when the user asks |
| Too few workers | tasks stay queued | raise `langgraph dev --n-jobs-per-worker` or deployment worker count |

---
## 9. When to use sync vs async

| Situation | Choice |
|-----------|--------|
| Simple delegation, under 10 seconds | synchronous subagent |
| Multi-minute research, coding, or builds | **async** |
| Multiple parallel tasks | **async** |
| Mid-flight update/cancel while chatting | **async** |
| Same-session tools only | synchronous |
| Separate service boundary | async + HTTP `url` |

---
## Summary

| Item | Takeaway |
|------|----------|
| Spec helper | `AsyncSubAgent(name, description, graph_id, url?, headers?)` returns a dict spec |
| Transport | ASGI co-deployment, HTTP remote, or hybrid |
| Five supervisor tools | start, check, update, cancel, list async tasks |
| State channel | dedicated `async_tasks` channel, separate from messages |
| Lifecycle | Launch → Check → Update → Cancel → List |
| Runtime | `langgraph dev --n-jobs-per-worker 10` or LangSmith Deployment |
| Prompt rules | no immediate polling, never truncate task IDs, assume stale status |
| Status | preview feature in `deepagents>=0.5.0` |

## Next steps
→ Use **LangSmith Studio** to monitor long-running background tasks.
→ Combine async subagents with the sandbox patterns from `10_sandboxes_and_acp.ipynb`.

---
**References:**
- `docs/deepagents/12-async-subagents.md`
- [Deep Agents async subagents](https://docs.langchain.com/oss/python/deepagents/async-subagents)
- [LangGraph local server](https://docs.langchain.com/oss/python/langgraph/local-server)